<a href="https://colab.research.google.com/github/adina-v/ASR-GL-Colab/blob/main/ASR_galego.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transcrición de audio en galego · Colab
Escolle un modelo, sube o audio e descarga a transcrición.

En Colab: **Entorno de execución → Cambiar tipo de entorno de execución → GPU**. Executa os seis pasos na orde indicada.

Descárgase un ZIP con **subtitulos.srt**, **transcricion_tempos.tsv**, **transcricion.txt** e **resultado.json**. Non se identifican os falantes, as linguas nin se crean clips de audio segmentados.


## 1. Preparar o contorno
Executa unha vez por sesión. Se xa importaches Transformers noutra cela, reinicia a sesión despois da instalación e continúa no paso 2.

In [2]:
#@title 1 · Instalar as ferramentas
import subprocess, sys, shutil
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.57.1', 'accelerate>=1.0,<2',
                'soundfile>=0.12,<1', 'librosa>=0.10.2,<0.12',
                'pandas>=2,<3', 'sentencepiece'], check=True)
if shutil.which('ffmpeg') is None:
    subprocess.run(['apt-get', '-qq', 'update'], check=True)
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
print('Instalación rematada. Continúa no paso 2.')


Instalación rematada. Continúa no paso 2.


## 2. Escoller o modelo
Cárgase un único modelo de cada vez. As duracións quedan como parámetros internos dos scripts.

Whisper: máximo 25 seguntos (s), mínimo procurado 5 s, busca de pausas nos últimos 10 s, pausa mínima 0,35 s e limiar RMS −35 dBFS. Entre pausas priorízanse as máis longas; sen pausa axeitada úsase un mínimo de enerxía. O audio non perde mostras. A puntuación e os segmentos son os producidos por Whisper, sen agrupación posterior por palabras.

CTC: anacos internos de 25 s e contexto de 6 s á esquerda e 3 s á dereita.


In [16]:
#@title 2 · Opcións
modelo = "Whisper turbo" #@param ["Whisper turbo", "Whisper LoS", "w2v-BERT", "wav2vec2"]
segundos_por_bloque = 25
feixes_whisper = 5
MODELOS = {
    'Whisper turbo': ('proxectonos/whisper-large-v3-turbo-gl-v1.0', 'whisper'),
    'Whisper LoS': ('BSC-LT/whisper-large-v3-LoS-punctuated', 'whisper'),
    'w2v-BERT': ('proxectonos/w2v-bert-2.0-gl', 'ctc'),
    'wav2vec2': ('proxectonos/Nos_ASR-wav2vec2-xls-r-300m-gl', 'ctc'),
}
assert segundos_por_bloque == 25
assert feixes_whisper in (1, 3, 5)
print('Seleccionado:', modelo)


Seleccionado: Whisper turbo


## 3. Seleccionar o audio
Sube un audio ou indica a súa ruta en Google Drive. Convértese a mono e 16 kHz con FFmpeg, mantendo velocidade e duración. Admítense WAV, MP3, FLAC, M4A e outros formatos que poida ler FFmpeg.


In [17]:
#@title 3 · Cargar un audio
orixe = "Subir ficheiro" #@param ["Subir ficheiro", "Google Drive"]
ruta_drive = "" #@param {type:"string"}
from pathlib import Path
from google.colab import files, drive
if orixe == 'Google Drive':
    drive.mount('/content/drive')
    audio_path = Path(ruta_drive).expanduser()
else:
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError('Selecciona un único ficheiro e volve executar esta cela.')
    audio_path = Path(next(iter(subidos))).resolve()
    del subidos
if not audio_path.is_file():
    raise FileNotFoundError(f'Non se atopa o audio: {audio_path}')
print('Audio seleccionado:', audio_path.name)



Saving ED-RUR-AGO-FENE-01-1993.wav to ED-RUR-AGO-FENE-01-1993.wav
Audio seleccionado: ED-RUR-AGO-FENE-01-1993.wav


## 4. Preparar o audio, o modelo e os intervalos
A descarga inicial pode tardar varios minutos.


In [18]:
#@title 4 · Preparar e cargar
import subprocess
import shutil
import gc, json, math, re, time, uuid, hashlib
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import soundfile as sf
import torch, transformers
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoModelForCTC, Wav2Vec2CTCTokenizer, Wav2Vec2FeatureExtractor, Wav2Vec2Processor, Wav2Vec2BertProcessor
from IPython.display import display, Audio
if not torch.cuda.is_available():
    raise RuntimeError('Activa unha GPU en Entorno de execución → Cambiar tipo de entorno de execución e volve executar as celas.')
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 6):
    raise RuntimeError('Este caderno require PyTorch 2.6 ou posterior para cargar pesos .bin. Usa un contorno de Colab actualizado.')
# Eliminar as referencias ao modelo anterior antes de cambiar de modelo.
for nome in ('asr', 'rede', 'procesador', 'loaded_ctc'):
    globals().pop(nome, None)
gc.collect()
torch.cuda.empty_cache()
traballo = Path('/content/asr_galego')
traballo.mkdir(exist_ok=True)
wav_path = traballo / 'audio_16k_mono.wav'
p = subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
                    '-i', str(audio_path), '-vn', '-ac', '1', '-ar', '16000',
                    '-c:a', 'pcm_s16le', str(wav_path)], capture_output=True, text=True)
if p.returncode:
    raise RuntimeError('Non se puido ler o audio: ' + p.stderr[-2000:])
info = sf.info(str(wav_path))
if info.frames < 1600:
    raise ValueError('O audio está baleiro ou dura menos de 0,1 segundos.')
repo, familia = MODELOS[modelo]
if modelo == 'wav2vec2':
    # Configuración
    tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(repo, unk_token='[UNK]',
                    pad_token='[PAD]', word_delimiter_token=' ')
    procesador = Wav2Vec2Processor(
        feature_extractor=Wav2Vec2FeatureExtractor.from_pretrained(repo), tokenizer=tokenizer)
elif modelo == 'w2v-BERT':
    procesador = Wav2Vec2BertProcessor.from_pretrained(repo)
else:
    procesador = AutoProcessor.from_pretrained(repo)
if familia == 'whisper':
    procesador.feature_extractor.return_attention_mask = True
clase = AutoModelForSpeechSeq2Seq if familia == 'whisper' else AutoModelForCTC
dtype_modelo = torch.float16 if familia == 'whisper' else torch.float32
rede = clase.from_pretrained(repo, torch_dtype=dtype_modelo, low_cpu_mem_usage=True).to('cuda')
rede.eval()
if familia == 'whisper':
    rede.generation_config.forced_decoder_ids = None
    procesador.tokenizer.set_prefix_tokens(language='gl', task='transcribe', predict_timestamps=True)
config_cargada = (modelo, repo, familia)
audio_cargado = str(audio_path)
print(f'GPU: {torch.cuda.get_device_name(0)} | Audio: {info.duration/60:.2f} minutos')
print('Modelo listo:', repo)
# Escoitar só os primeiros 20 segundos, sen cargar todo o audio.
with sf.SoundFile(wav_path) as f:
    mostra = f.read(min(20*16000, len(f)), dtype='float32')
display(Audio(mostra, rate=16000))

# Funcións de segmentación
from typing import Iterable
TARGET_SR = 16000
def frame_rms_dbfs(
    audio: np.ndarray,
    frame_seconds: float = 0.01,
) -> tuple[np.ndarray, np.ndarray]:
    """Calcula enerxía RMS en dBFS e o centro temporal de cada trama."""

    frame_samples = max(1, int(round(frame_seconds * TARGET_SR)))
    if len(audio) < frame_samples:
        rms = float(np.sqrt(np.mean(np.square(audio, dtype=np.float64))))
        return (
            np.asarray([20.0 * np.log10(max(rms, 1e-10))]),
            np.asarray([len(audio) / (2.0 * TARGET_SR)]),
        )

    frame_count = len(audio) // frame_samples
    frames = audio[: frame_count * frame_samples].reshape(frame_count, frame_samples)
    # ``einsum`` acumula en float64 sen crear unha copia float64 de todo o
    # audio; isto mantén baixo o uso de memoria tamén en gravacións longas.
    sums = np.einsum("ij,ij->i", frames, frames, dtype=np.float64)
    rms = np.sqrt(sums / frame_samples)
    dbfs = 20.0 * np.log10(np.maximum(rms, 1e-10))
    centers = (
        np.arange(frame_count, dtype=np.float64) * frame_samples
        + frame_samples / 2.0
    ) / TARGET_SR
    return dbfs, centers

def choose_pause_boundary(
    dbfs: np.ndarray,
    centers: np.ndarray,
    segment_start: float,
    target_end: float,
    min_segment_seconds: float,
    pause_search_seconds: float,
    min_pause_seconds: float,
    silence_threshold_dbfs: float,
) -> float:
    """Escolle unha pausa antes do límite ou, como último recurso, o mínimo RMS."""

    search_start = max(
        segment_start + min_segment_seconds,
        target_end - pause_search_seconds,
    )
    indices = np.flatnonzero((centers >= search_start) & (centers <= target_end))
    if not len(indices):
        return target_end

    quiet = dbfs[indices] <= silence_threshold_dbfs
    minimum_frames = max(1, int(np.ceil(min_pause_seconds / 0.01)))
    runs: list[tuple[int, int]] = []
    run_start: int | None = None

    for position, is_quiet in enumerate(quiet):
        if is_quiet and run_start is None:
            run_start = position
        elif not is_quiet and run_start is not None:
            if position - run_start >= minimum_frames:
                runs.append((run_start, position - 1))
            run_start = None
    if run_start is not None and len(quiet) - run_start >= minimum_frames:
        runs.append((run_start, len(quiet) - 1))

    if runs:
        candidates: list[tuple[float, float]] = []
        search_duration = max(target_end - search_start, 1e-6)
        for first, last in runs:
            run_indices = indices[first : last + 1]
            duration = (centers[run_indices[-1]] - centers[run_indices[0]]) + 0.01
            midpoint = float(
                (centers[run_indices[0]] + centers[run_indices[-1]]) / 2.0
            )
            # Priorizamos pausas claras e longas, cunha preferencia secundaria
            # polas máis próximas ao límite para non crear segmentos moi curtos.
            closeness = (midpoint - search_start) / search_duration
            candidates.append((duration + 0.20 * closeness, midpoint))
        return max(candidates)[1]

    # Se non hai unha pausa que cumpra a duración mínima, o corte faise no
    # último tramo situada a 1 dB do mínimo de enerxía dispoñible. Así se
    # favorece un segmento longo cando varios puntos son acusticamente iguais.
    # Non se descarta ningunha mostra.
    local_dbfs = dbfs[indices]
    near_minimum = indices[local_dbfs <= float(np.min(local_dbfs)) + 1.0]
    quietest = near_minimum[-1]
    return float(centers[quietest])

def audio_windows(
    audio: np.ndarray,
    max_seconds: float,
    min_segment_seconds: float,
    pause_search_seconds: float,
    min_pause_seconds: float,
    silence_threshold_dbfs: float,
) -> Iterable[tuple[float, float, np.ndarray]]:
    """Crea ventás continuas de ata 30 s, preferentemente limitadas por pausas."""

    total_samples = len(audio)
    total_seconds = total_samples / TARGET_SR
    dbfs, centers = frame_rms_dbfs(audio)
    sample_start = 0

    while sample_start < total_samples:
        segment_start = sample_start / TARGET_SR
        target_end = min(segment_start + max_seconds, total_seconds)

        if target_end >= total_seconds:
            sample_end = total_samples
        else:
            boundary = choose_pause_boundary(
                dbfs=dbfs,
                centers=centers,
                segment_start=segment_start,
                target_end=target_end,
                min_segment_seconds=min_segment_seconds,
                pause_search_seconds=pause_search_seconds,
                min_pause_seconds=min_pause_seconds,
                silence_threshold_dbfs=silence_threshold_dbfs,
            )
            sample_end = int(round(boundary * TARGET_SR))
            maximum_end = min(
                sample_start + int(round(max_seconds * TARGET_SR)),
                total_samples,
            )
            sample_end = min(max(sample_end, sample_start + 1), maximum_end)

        yield (
            sample_start / TARGET_SR,
            sample_end / TARGET_SR,
            audio[sample_start:sample_end],
        )
        sample_start = sample_end

# O audio 16 kHz queda en RAM de CPU; só se envía unha xanela á GPU.
onda_completa, _ = sf.read(wav_path, dtype='float32')
if not np.isfinite(onda_completa).all():
    raise ValueError('O audio contén valores non finitos.')
fragmentos = [(a,b) for a,b,_ in audio_windows(
    onda_completa, max_seconds=30, min_segment_seconds=5,
    pause_search_seconds=10, min_pause_seconds=0.35,
    silence_threshold_dbfs=-35)]
print(f'Preparados {len(fragmentos)} intervalos.')


preprocessor_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

... (3 outputs left out)

model.safetensors:   0%|          | 0.00/3.24G [00:00<?, ?B/s]

generation_config.json: 0.00B [00:00, ?B/s]

GPU: Tesla T4 | Audio: 18.83 minutos
Modelo listo: proxectonos/whisper-large-v3-turbo-gl-v1.0


Preparados 45 intervalos.


## 5. Transcribir
Cada bloque rematado gárdase nun rexistro parcial. Se a execución se interrompe, a cela 6 permite descargar a parte terminada. Reexecutar a cela 5 comeza unha execución nova.


In [19]:
#@title 5 · Transcribir o audio
from tqdm.auto import tqdm
if config_cargada[0] != modelo or audio_cargado != str(audio_path):
    raise RuntimeError('Cambiaches o modelo ou o audio: volve executar a cela 4.')
_, repo, familia = config_cargada
nome_seguro = re.sub(r'[^\w.-]+', '_', audio_path.stem)[:80] or 'audio'
run_dir = traballo / (nome_seguro + '_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:6])
run_dir.mkdir()
filas, textos = [], []
completo = False
procesado_ata = 0.0
inicio_execucion = time.monotonic()
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
metadatos = {
    'audio': audio_path.name, 'modelo': repo, 'familia': familia,
    'revision_modelo': getattr(rede.config, '_commit_hash', None),
    'data_utc': datetime.now(timezone.utc).isoformat(),
    'duracion_audio': info.duration, 'sampling_rate': 16000,
    'segundos_por_bloque': segundos_por_bloque, 'solapamento': 0, 'segmentacion': 'RMS script orixinal: max 30 s, busca 10 s, min 5 s, pausa 0.35 s, -35 dBFS',
    'ctc_chunk_seconds': 30 if familia == 'ctc' else None,
    'ctc_context_seconds': [6, 3] if familia == 'ctc' else None,
    'whisper_max_new_tokens': 256 if familia == 'whisper' else None,
    'whisper_temperature_fallback': True if familia == 'whisper' else None,
    'feixes_whisper': feixes_whisper if familia == 'whisper' else None,
    'lingua_whisper': 'gl' if familia == 'whisper' else None,
    'dtype': str(dtype_modelo), 'transformers': transformers.__version__,
    'torch': torch.__version__, 'gpu': torch.cuda.get_device_name(0),
    'posprocesamento': {'whisper': 'eliminar ¿ e ¡', 'w2v-BERT': 'ng[UNK] → ngü; [UNK]non → non; sen distinguir caixa'},
    'modo_ctc': 'greedy sen LM' if familia == 'ctc' else None,
}

def posprocesar(texto, modelo_nome):
    if modelo_nome in ('Whisper turbo', 'Whisper LoS'):
        return texto.replace('¿', '').replace('¡', '').strip()
    if modelo_nome == 'w2v-BERT':
        texto = re.sub(r'(ng)\[unk\]', lambda m: m.group(1) + 'ü', texto, flags=re.IGNORECASE)
        texto = re.sub(r'\[unk\](non)', r'\1', texto, flags=re.IGNORECASE)
    return texto.strip()

def segmentos_whisper(decoded, inicio, fin):
    # Mesmo principio ca decoded_segments do script; engádese trazabilidade.
    texto_raw = ' '.join((decoded if isinstance(decoded, str) else decoded.get('text', '')).split())
    offsets = [] if isinstance(decoded, str) else decoded.get('offsets', []) or []
    rows = []
    for off in offsets:
        raw = ' '.join(str(off.get('text', '')).split())
        ts = off.get('timestamp') or off.get('timestamps')
        if not raw or ts is None or len(ts) != 2:
            continue
        a, b = ts
        reparado = a is None or b is None
        a = 0.0 if a is None else float(a)
        b = fin-inicio if b is None else float(b)
        if not math.isfinite(a) or not math.isfinite(b):
            continue
        aa = min(max(a,0),fin-inicio)
        bb = min(max(b,aa),fin-inicio)
        reparado = reparado or aa != a or bb != b
        if bb > aa:
            rows.append({'start_time': inicio+aa, 'end_time': inicio+bb,
                         'text_raw': raw, 'text': posprocesar(raw, modelo),
                         'tipo_tempo': 'segmento_whisper_axustado' if reparado else 'segmento_whisper',
                         'status': 'revisar aliñamento' if reparado else 'ok'})
    # Se offsets incompletos perderían palabras, gardar o texto íntegro.
    normal = lambda t: ''.join(t.split())
    if texto_raw and (not rows or normal(''.join(r['text_raw'] for r in rows)) != normal(texto_raw)):
        rows = [{'start_time': inicio, 'end_time': fin, 'text_raw': texto_raw,
                 'text': posprocesar(texto_raw, modelo), 'tipo_tempo': 'xanela_audio_respaldo',
                 'status': 'revisar aliñamento'}]
    return texto_raw, rows


def _move_inputs_to_device(inputs, device: torch.device) -> dict:
    return {
        key: value.to(device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

def _model_align_to(model) -> int:
    return int(getattr(model.config, "inputs_to_logits_ratio", 1) or 1)

def _rescale_stride_one(stride, ratio: float):
    """
    Convert an audio-sample stride tuple into token/logit-frame units.

    This mirrors the relevant stride rescaling used by the HF ASR pipeline for
    plain CTC models.
    """
    input_n, left, right = stride
    token_n = int(round(input_n * ratio))
    left_n = int(round(left / input_n * token_n)) if input_n else 0
    right_n = int(round(right / input_n * token_n)) if input_n else 0
    return token_n, left_n, right_n

def _ctc_get_tokenizer(loaded_model: dict):
    if loaded_model.get("tokenizer") is not None:
        return loaded_model["tokenizer"]

    processor = loaded_model.get("processor")
    if processor is not None and hasattr(processor, "tokenizer"):
        return processor.tokenizer

    raise RuntimeError("CTC tokenizer not available in loaded model record.")

def _ctc_get_feature_extractor(loaded_model: dict):
    if loaded_model.get("feature_extractor") is not None:
        return loaded_model["feature_extractor"]

    processor = loaded_model.get("processor")
    if processor is not None and hasattr(processor, "feature_extractor"):
        return processor.feature_extractor

    if processor is not None:
        return processor

    raise RuntimeError("CTC feature extractor not available in loaded model record.")

def _ctc_forward_token_ids(loaded_model: dict, audio: np.ndarray) -> torch.Tensor:
    """
    Forward one CTC chunk and return greedy token IDs.

    This avoids processor.batch_decode() during chunk recombination. The final
    decode happens once after overlap has been cropped in token-frame space.
    """
    feature_extractor = _ctc_get_feature_extractor(loaded_model)
    model = loaded_model["model"]
    device = loaded_model["device"]
    sampling_rate = loaded_model["sampling_rate"]

    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    try:
        inputs = feature_extractor(
            audio,
            sampling_rate=sampling_rate,
            return_tensors="pt",
            return_attention_mask=True,
        )
    except TypeError:
        inputs = feature_extractor(
            audio,
            sampling_rate=sampling_rate,
            return_tensors="pt",
        )

    inputs = _move_inputs_to_device(inputs, device)

    with torch.inference_mode():
        logits = model(**inputs).logits
        if not torch.isfinite(logits).all().item():
            raise RuntimeError('O modelo produciu valores non finitos.')
        token_ids = logits.argmax(dim=-1).detach().cpu()
        del logits

    del inputs

    if isinstance(device, torch.device) and device.type == "cuda":
        torch.cuda.empty_cache()

    return token_ids

def _transcribe_ctc_pipeline_emulated(loaded_model: dict, audio: np.ndarray) -> str:
    """
    Emulate Hugging Face AutomaticSpeechRecognitionPipeline for plain CTC models.

    """
    model = loaded_model["model"]
    tokenizer = _ctc_get_tokenizer(loaded_model)
    sampling_rate = int(loaded_model.get("sampling_rate", 16000))
    options = loaded_model.get("options", {}) or {}

    audio = np.asarray(audio, dtype=np.float32).reshape(-1)

    chunk_length_s = float(options.get("chunk_length_s", 30.0))
    stride_left_s = float(options.get("stride_left_s", 6.0))
    stride_right_s = float(options.get("stride_right_s", 3.0))
    skip_special_tokens = bool(options.get("decode_skip_special_tokens", False))

    align_to = _model_align_to(model)

    chunk_len = int(round(chunk_length_s * sampling_rate / align_to) * align_to)
    stride_left = int(round(stride_left_s * sampling_rate / align_to) * align_to)
    stride_right = int(round(stride_right_s * sampling_rate / align_to) * align_to)

    if chunk_len <= 0:
        raise ValueError(f"Invalid chunk_length_s: {chunk_length_s}")

    if chunk_len <= stride_left + stride_right:
        raise ValueError(
            "Chunk length must be greater than stride length: "
            f"chunk={chunk_length_s}, stride_left={stride_left_s}, "
            f"stride_right={stride_right_s}"
        )

    step = chunk_len - stride_left - stride_right
    inputs_len = audio.shape[0]

    print(
        f"[transcribe] Emulated HF CTC pipeline {loaded_model.get('model_id')}: "
        f"{inputs_len / float(sampling_rate):.2f}s audio, "
        f"{chunk_length_s:.2f}s chunks, "
        f"stride=({stride_left_s:.2f}s,{stride_right_s:.2f}s), "
        f"align_to={align_to}, "
        f"decode_skip_special_tokens={skip_special_tokens}"
    )

    final_items = []
    total_chunks = 0

    for chunk_start_idx in range(0, inputs_len, step):
        chunk_end_idx = chunk_start_idx + chunk_len
        chunk = audio[chunk_start_idx:chunk_end_idx]

        if chunk.shape[0] == 0:
            continue

        is_first = chunk_start_idx == 0
        is_last = chunk_end_idx >= inputs_len

        current_stride_left = 0 if is_first else stride_left
        current_stride_right = 0 if is_last else stride_right

        total_chunks += 1

        print(
            f"[transcribe] {loaded_model.get('model_id')}: "
            f"emulated chunk {total_chunks} "
            f"start={chunk_start_idx / float(sampling_rate):.2f}s "
            f"end={min(chunk_end_idx, inputs_len) / float(sampling_rate):.2f}s "
            f"chunk={chunk.shape[0] / float(sampling_rate):.2f}s"
        )

        token_ids = _ctc_forward_token_ids(loaded_model, chunk)

        # BERT recibe espectrogramas: obter a escala real das tramas de saída.
        ratio = (token_ids.shape[-1] / float(chunk.shape[0])
                 if loaded_model['model_type'] == 'w2v_bert'
                 else 1.0 / float(align_to))
        total_n, left_n, right_n = _rescale_stride_one(
            (
                int(chunk.shape[0]),
                int(current_stride_left),
                int(current_stride_right),
            ),
            ratio,
        )

        items = token_ids.numpy()
        right_boundary = max(left_n, total_n - right_n)
        cropped = items[:, left_n:right_boundary]

        if cropped.shape[1] > 0:
            final_items.append(cropped)

        del token_ids

        if is_last:
            break

    if not final_items:
        return ""

    merged = np.concatenate(final_items, axis=1).squeeze(0)

    text = tokenizer.decode(
        merged,
        skip_special_tokens=skip_special_tokens,
    )

    print(
        f"[transcribe] Emulated HF CTC pipeline decoded "
        f"{len(final_items)} chunks into {merged.shape[0]} token frames."
    )

    return text.strip()

if familia == 'ctc':
    loaded_ctc = {'model': rede, 'processor': procesador,
                  'tokenizer': procesador.tokenizer,
                  'feature_extractor': procesador.feature_extractor,
                  'device': torch.device('cuda'), 'sampling_rate': 16000,
                  'model_id': modelo,
                  'model_type': 'w2v_bert' if modelo == 'w2v-BERT' else 'wav2vec2',
                  'options': {'chunk_length_s': 30.0, 'stride_left_s': 6.0,
                              'stride_right_s': 3.0, 'decode_skip_special_tokens': False}}

def transcribir_ctc(onda):
    return _transcribe_ctc_pipeline_emulated(loaded_ctc, onda)

with sf.SoundFile(wav_path) as audio, open(run_dir / 'bloques_parciais.jsonl', 'w', encoding='utf-8') as parcial:
    for bloque, (inicio_previsto, fin_previsto) in enumerate(tqdm(fragmentos, desc='Transcribindo', unit='bloque')):
        primeiro = round(inicio_previsto * 16000)
        ultimo = min(info.frames, round(fin_previsto * 16000))
        audio.seek(primeiro)
        inicio = primeiro / 16000
        onda = audio.read(ultimo-primeiro, dtype='float32')
        fin = inicio + len(onda) / 16000
        filas_bloque = []
        try:
            with torch.inference_mode():
                if familia == 'whisper':
                    entradas = procesador([onda], sampling_rate=16000, return_tensors='pt',
                                          truncation=False, padding='longest', return_attention_mask=True)
                    ids = rede.generate(
                        input_features=entradas.input_features.to('cuda',dtype=torch.float16),
                        attention_mask=entradas.attention_mask.to('cuda'),
                        language='gl', task='transcribe', return_timestamps=True,
                        condition_on_prev_tokens=True, num_beams=feixes_whisper,
                        max_new_tokens=256, temperature=(0.0,0.2,0.4,0.6,0.8,1.0),
                        compression_ratio_threshold=1.35, logprob_threshold=-1.0,
                        no_speech_threshold=0.6)
                    decoded = procesador.tokenizer.decode(ids[0], skip_special_tokens=True,
                                                         output_offsets=True, time_precision=0.02)
                    texto_raw, filas_bloque = segmentos_whisper(decoded,inicio,fin)
                    texto = posprocesar(texto_raw,modelo)
                    del entradas,ids
                else:
                    texto_raw = transcribir_ctc(onda)
                    texto = posprocesar(texto_raw, modelo)
                    # Conservar tamén intervalos que produzan texto baleiro.
                    filas_bloque.append({'start_time': inicio_previsto, 'end_time': fin_previsto,
                                        'text': texto, 'text_raw': texto_raw,
                                        'tipo_tempo': 'xanela_audio_ctc',
                                        'status': 'ok' if texto else 'texto_baleiro'})
        except torch.cuda.OutOfMemoryError as exc:
            torch.cuda.empty_cache()
            raise RuntimeError('Memoria GPU insuficiente. Podes exportar a parte feita na cela 6. Logo reinicia a sesión e escolle un modelo máis pequeno.') from exc
        for fila in filas_bloque:
            fila.update({'block_id': bloque+1, 'block_start': inicio, 'block_end': fin})
        parcial.write(json.dumps({'block_id': bloque+1, 'start': inicio, 'end': fin,
                                  'text': texto, 'text_raw': texto_raw, 'rows': filas_bloque,
                                  'whisper_offsets': decoded if familia == 'whisper' else None}, ensure_ascii=False)+'\n')
        parcial.flush()
        filas.extend(filas_bloque)
        textos.append(texto)
        procesado_ata = fin
completo = True
print(f'Rematado en {(time.monotonic() - inicio_execucion) / 60:.1f} minutos.')

columnas = ["start_time", "end_time", "text", "status", "block_id"]
display(pd.DataFrame(filas, columns=columnas).head(20))


Transcribindo:   0%|          | 0/45 [00:00<?, ?bloque/s]

`generation_config` default values have been modified to match model-specific defaults: {'suppress_tokens': [], 'begin_suppress_tokens': [220, 50257]}. If this is not desired, please set these values explicitly.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProce

Rematado en 1.3 minutos.


,start_time,end_time,text,status,block_id
0,0.000,24.135,os galegos. Luís Unqueiro era un home que case...,revisar aliñamento,1
1,24.135,49.205,"o. Eu lle tomaba o pelo moitas veces. E, nunha...",revisar aliñamento,2
2,49.205,76.075,tiña un cordel no de Cinturón e tiña un cuchil...,revisar aliñamento,3
3,76.075,105.140,"e decía que era o río de que el, que era do rí...",revisar aliñamento,4
4,105.140,126.425,"e lle di e entonces lle digo eu, que non me ac...",revisar aliñamento,5
5,126.425,148.350,o tío: 'Xa a contei'. 'Non a contei'. Entonces...,revisar aliñamento,6
6,148.350,174.285,". Entonces, el volveu: 'Non, que cha contei'. ...",revisar aliñamento,7
7,174.285,201.285,". E é bastante amigo meu. Entonces, está Juan ...",revisar aliñamento,8
8,201.285,228.475,"E el dixen eu: 'Bueno, ti estás enfadado con t...",revisar aliñamento,9
9,228.475,256.580,"o. Non paro aí, no no dialecto del, que ten qu...",revisar aliñamento,10


## 6. Descargar
O ZIP contén catro ficheiros:

- **transcricion.txt:** texto continuo.
- **transcricion_tempos.tsv:** segmentos con tempos e texto, nas sete columnas do formato de corpus. Os nomes .wav da columna path son identificadores; non se crean clips. speaker=unknown indica que non hai diarización.
- **subtitulos.srt:** o mesmo texto segmentado con tempos en formato de subtítulos.

- **resultado.json:** texto completo, segmentos co texto orixinal e corrixido, procedencia dos tempos, estado da aliñación e metadatos da execución.

Se interrompes a transcrición, podes descargar a parte rematada; o nome do ZIP levará PARCIAL. Descarga o ZIP antes de pechar a sesión.


In [20]:
#@title 6 · Descargar os resultados
import json
import shutil
import zipfile
from pathlib import Path

import pandas as pd
from google.colab import files, drive

copiar_a_drive = False #@param {type:"boolean"}
cartafol_drive = "/content/drive/MyDrive/Transcricions_ASR" #@param {type:"string"}

if "run_dir" not in globals():
    raise RuntimeError("Executa primeiro a cela 5.")

rexistros = [
    json.loads(liña)
    for liña in (run_dir / "bloques_parciais.jsonl")
    .read_text(encoding="utf-8").splitlines()
    if liña.strip()
]

columnas = ["start_time", "end_time", "text", "status", "block_id"]

filas_exportar = [
    {columna: segmento.get(columna) for columna in columnas}
    for rexistro in rexistros
    for segmento in rexistro["rows"]
]

# Texto continuo co posprocesamento aplicado.
texto_completo = " ".join(
    rexistro["text"] for rexistro in rexistros if rexistro["text"]
)
(run_dir / "transcricion.txt").write_text(
    texto_completo + "\n", encoding="utf-8"
)

# Tempos en formato SRT.
def reloxo(segundos):
    ms = round(float(segundos) * 1000)
    h, resto = divmod(ms, 3600000)
    m, resto = divmod(resto, 60000)
    s, ms = divmod(resto, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


srt = []
for fila in filas_exportar:
    a, b = fila["start_time"], fila["end_time"]
    if (
        a is not None
        and b is not None
        and b > a
        and round(b * 1000) > round(a * 1000)
    ):
        texto = " ".join(fila["text"].split())
        srt.append(
            f"{len(srt) + 1}\n"
            f"{reloxo(a)} --> {reloxo(b)}\n"
            f"{texto}\n"
        )

(run_dir / "subtitulos.srt").write_text(
    "\n".join(srt), encoding="utf-8"
)

# TSV: só as cinco columnas seleccionadas.
pd.DataFrame(filas_exportar, columns=columnas).to_csv(
    run_dir / "transcricion_tempos.tsv",
    sep="\t",
    index=False,
    encoding="utf-8",
)

# JSON: só os segmentos cos mesmos cinco campos.
(run_dir / "resultado.json").write_text(
    json.dumps(filas_exportar, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

# Descargar un ZIP cos catro ficheiros.
zip_path = str(run_dir) + (".zip" if completo else "_PARCIAL.zip")

with zipfile.ZipFile(
    zip_path, "w", compression=zipfile.ZIP_DEFLATED
) as z:
    for nome in (
        "subtitulos.srt",
        "transcricion_tempos.tsv",
        "transcricion.txt",
        "resultado.json",
    ):
        z.write(run_dir / nome, arcname=nome)

if copiar_a_drive:
    drive.mount("/content/drive")
    destino = Path(cartafol_drive)
    destino.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, destino / Path(zip_path).name)
    print("Copia gardada en:", destino)

print("Resultado COMPLETO" if completo else "ATENCIÓN: resultado PARCIAL")
print(
    "Filas omitidas dos subtítulos por tempos incompletos:",
    len(filas_exportar) - len(srt),
)

axustadas = sum(
    fila["status"] == "revisar aliñamento"
    for fila in filas_exportar
)

files.download(zip_path)

Resultado COMPLETO
Filas omitidas dos subtítulos por tempos incompletos: 0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Reutilización
Para cambiar de modelo ou audio, repite os pasos 2–6. Mantense Whisper con cinco feixes, sen selector de duración nin de feixes.

Se falta memoria GPU, descarga a parte rematada, reinicia a sesión e proba un modelo máis pequeno. A puntuación e os límites de frase poden requirir revisión. En CTC non se restaura a puntuación automaticamente.
